# LED frequency
## Assumptions & design choices
Assume maximum LED count on is 164 per `brightness` buffer.
- This allows us update LEDs more frequently rather than having a fixed time step for going through all 256 LEDs.
- The downside is flicker may be visible for LED counts exceeding that value.

Change timer dynamically based on the brightness of the next led.
- This allows 4 discrete levels of brightness and keeps the cost around the same as
  having one brightness at the brightness level 2-3.



In [33]:
import numpy as np

F_CPU = 8_000_000
N_LEDS = 164  # Estimated maximum lit LED count. This is briefly surpassed on shaking.
FPS = 19

# Frequency based on LED ON times

In [34]:

def calculate_timings(T_leds_on_us, T_on_weights):

    weights = np.array(T_on_weights, dtype=float)

    # Time ON distribution for brightness levels 0-3.
    # Level 0 is skipped so we exclude it.
    T_led = np.array(T_leds_on_us) * 1e-6                  # μs -> s
    T_led_avg = np.average(T_led, weights=weights)  # s

    T_frame = N_LEDS * T_led  # s
    T_frame_avg = N_LEDS * T_led_avg             # s

    print(f"LED ON time: {(T_led * 1e6).round(3)} μs")
    print(f"  weighted avg: {(T_led_avg * 1e6).round(decimals=3)} μs")
    print(f"Frame time based on LED ON time: {(T_frame * 1e3).round(3)} ms")
    print(f"  weighted avg: {(T_frame_avg * 1e3).round(3)} ms")

    print()

    f_led = 1 / T_led                       # Hz
    f_led_avg = 1 / T_led_avg      # Hz

    f_frame = 1 / T_frame                   # Hz
    f_frame_avg = 1 / T_frame_avg  # Hz

    print(f"LED frequencies: {f_led.round(3)} Hz")
    print(f"  weighted avg: {f_led_avg.round(decimals=3)} Hz")
    print(f"Frame frequencies based on LED freq: {f_frame.round(3)} Hz")
    print(f"  weighted avg: {f_frame_avg.round(3)} Hz")

    return f_led

T_leds_on = [10, 20, 40]
weights = [0.27, 0.13, 0.60]  # From experimantation
calculate_timings(T_leds_on, weights)


LED ON time: [10. 20. 40.] μs
  weighted avg: 29.3 μs
Frame time based on LED ON time: [1.64 3.28 6.56] ms
  weighted avg: 4.805 ms

LED frequencies: [100000.  50000.  25000.] Hz
  weighted avg: 34129.693 Hz
Frame frequencies based on LED freq: [609.756 304.878 152.439] Hz
  weighted avg: 208.108 Hz


array([100000.,  50000.,  25000.])

The simulation uses around 250 000 cycles/simulation frame. The simulation FPS is 19 so if we split the simulation into 19 intervals, 1 interval = 421 000 cycles, our budget left for LEDs is 171 000 cycles. At 8 MHz that takes 171 000 / 8 000 000 = 0.021375 s ≈ 21 ms giving us the limit for our frame time.

The LED frequencies [100000 50000 25000] Hz for LED ON time: [10 20 40] μs are way too high for our code so we need to increase the LED ON times. The code 

In [35]:
T_leds_on = [50, 100, 200]
calculate_timings(T_leds_on, weights)

LED ON time: [ 50. 100. 200.] μs
  weighted avg: 146.5 μs
Frame time based on LED ON time: [ 8.2 16.4 32.8] ms
  weighted avg: 24.026 ms

LED frequencies: [20000. 10000.  5000.] Hz
  weighted avg: 6825.939 Hz
Frame frequencies based on LED freq: [121.951  60.976  30.488] Hz
  weighted avg: 41.622 Hz


array([20000., 10000.,  5000.])

# Maximum LED interrupt frequency budget from LED interrupt ICR cycles

The following code calculates the ceiling for how often the LED interrupt can run at maximum. It depends on the cycles used per LED interrupt.

In [36]:
# Prefixes:
# C = cycles
# F = cycle frequency

# Cycles a single led interrupt costs.
# This determines how often we can update LEDs.
# C_led_interrupt = 200

# Cycle budget per frame
C_frame = F_CPU * 1 / FPS
C_sim_frame = 250_000 # Probably more like 240 000.

def calculate_max_led_interrupt_frequency(C_led_interrupt):
    CF_leds = FPS * (C_frame - C_sim_frame)

    print(f"Cycle usage by simulation: {C_sim_frame * FPS} cycle/s")
    print(f"Cycle budget for LEDs:     {CF_leds} cycle/s")


    NF_led_interrupts = CF_leds / C_led_interrupt

    print(f"LED interrupts:     {NF_led_interrupts} Hz")

    NF_frame = NF_led_interrupts / N_LEDS

    print(f"Full frame updates: {round(NF_frame, 3)} Hz")

    T_frame = 1 / NF_frame

    print(f"Time per full frame update: {round(T_frame, 5)} s")

    T_led = T_frame / N_LEDS

    print(f"Time per led update: {T_led * 1e6} μs")

    F_led = 1 / T_led

    print(f"Maximum LED interrupt frequency: {round(F_led, 3)} Hz")


calculate_max_led_interrupt_frequency(200)


Cycle usage by simulation: 4750000 cycle/s
Cycle budget for LEDs:     3250000.0 cycle/s
LED interrupts:     16250.0 Hz
Full frame updates: 99.085 Hz
Time per full frame update: 0.01009 s
Time per led update: 61.53846153846155 μs
Maximum LED interrupt frequency: 16250.0 Hz


Based on the previous results we need to make the lowest brightness slightly brighter:

In [37]:
T_leds_on = [65, 100, 200]
f_int = calculate_timings(T_leds_on, weights)

LED ON time: [ 65. 100. 200.] μs
  weighted avg: 150.55 μs
Frame time based on LED ON time: [10.66 16.4  32.8 ] ms
  weighted avg: 24.69 ms

LED frequencies: [15384.615 10000.     5000.   ] Hz
  weighted avg: 6642.312 Hz
Frame frequencies based on LED freq: [93.809 60.976 30.488] Hz
  weighted avg: 40.502 Hz



# OCR0 values for LED update interrupts

$$f_{int}=\frac{F_{CPU}}{N\left(OCR0+1\right)} \quad \Longrightarrow \quad OCR0=\frac{F_{CPU}}{Nf_{int}}-1,$$

where $N\in\{1, 8, 64, 256, 1024\}$ is the prescaler.

In [38]:
N = 8     # Prescaler
OCR0 = F_CPU / (N * f_int) - 1

print(f"OCR0 values for interrupt frequencies at prescaler={N}:")
print(f"  Frequencies: {f_int.round(3)}")
print(f"  OCR0:        {np.ceil(OCR0).astype(int)}")  # Round up to be safer
print(f"  LEDs ON time: {T_leds_on}")

OCR0 values for interrupt frequencies at prescaler=8:
  Frequencies: [15384.615 10000.     5000.   ]
  OCR0:        [ 64  99 199]
  LEDs ON time: [65, 100, 200]


The final OCR0 values will be [64, 99, 199].

## Updated OCR0 values
Optimizations in the LED code freed up 50-80 cycles per ISR.
Let's try to pump up the frequencies slightly.

In [42]:
FPS = 20

calculate_max_led_interrupt_frequency(140)

Cycle usage by simulation: 5000000 cycle/s
Cycle budget for LEDs:     3421052.6315789474 cycle/s
LED interrupts:     24436.09022556391 Hz
Full frame updates: 149.001 Hz
Time per full frame update: 0.00671 s
Time per led update: 40.923076923076934 μs
Maximum LED interrupt frequency: 24436.09 Hz


In [43]:
dt = -15
T_leds_on = [65+dt, 100+dt, 200+dt]
f_int = calculate_timings(T_leds_on, weights)

LED ON time: [ 50.  85. 185.] μs
  weighted avg: 135.55 μs
Frame time based on LED ON time: [ 8.2  13.94 30.34] ms
  weighted avg: 22.23 ms

LED frequencies: [20000.    11764.706  5405.405] Hz
  weighted avg: 7377.352 Hz
Frame frequencies based on LED freq: [121.951  71.736  32.96 ] Hz
  weighted avg: 44.984 Hz


In [44]:
N = 8     # Prescaler
OCR0 = F_CPU / (N * f_int) - 1

print(f"OCR0 values for interrupt frequencies at prescaler={N}:")
print(f"  Frequencies: {f_int.round(3)}")
print(f"  OCR0:        {np.ceil(OCR0).astype(int)}")  # Round up to be safer
print(f"  LEDs ON time: {T_leds_on}")

OCR0 values for interrupt frequencies at prescaler=8:
  Frequencies: [20000.    11764.706  5405.405]
  OCR0:        [ 49  84 184]
  LEDs ON time: [50, 85, 185]
